# Few-Shot vs. Chain-of-Thought (CoT) Reasoning

**Testing with Real LLM (Ollama + LangChain)**

**Problem**: Last Letter Concatenation - Take the last letter of each word and concatenate them.

Examples:
- "Elon Musk" → "nk"
- "Bill Gates" → "ls"
- "Barack Obama" → "ka"

In [ ]:
from langchain_ollama import OllamaLLM
import re

# Use a smaller model to see the difference more clearly
# Larger models may perform well on both due to instruction tuning
llm = OllamaLLM(model="llama3.1:8b", temperature=0)

def get_correct_answer(name: str) -> str:
    """Ground truth: concatenate last letter of each word."""
    return ''.join(word[-1] for word in name.split())

# Test cases
test_names = [
    "Barack Obama",
    "Taylor Swift", 
    "Leonardo DiCaprio",
    "Marie Curie",
    "Albert Einstein",
    "Steve Jobs",
    "Mark Zuckerberg",
    "Oprah Winfrey",
    "Warren Buffett",
    "Serena Williams"
]

print("LLM initialized: llama3.1:8b")
print(f"Test cases: {len(test_names)} names")

LLM initialized: llama3.2:1b
Test cases: 10 names


---
## Part 2: Few-Shot Prompting

Few-shot prompting provides input-output examples **without** explaining the reasoning process.

The model must infer the pattern from examples alone.

In [2]:
def few_shot_prompt(query: str) -> str:
    """Create a few-shot prompt with only input-output examples."""
    return '''Q: "Elon Musk"
A: "nk"

Q: "Bill Gates"
A: "ls"

Q: "''' + query + '''"
A: "'''

print("Few-Shot Prompt Template:")
print("-" * 40)
print(few_shot_prompt("[NAME]"))

Few-Shot Prompt Template:
----------------------------------------
Q: "Elon Musk"
A: "nk"

Q: "Bill Gates"
A: "ls"

Q: "[NAME]"
A: "


In [3]:
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 60)

few_shot_results = []
for name in test_names:
    prompt = few_shot_prompt(name)
    response = llm.invoke(prompt).strip()[:50]
    correct = get_correct_answer(name)
    
    # Extract the first word/letters from response
    letters = re.findall(r'[a-zA-Z]+', response)
    answer = letters[0].lower() if letters else ""
    
    is_correct = answer == correct
    few_shot_results.append(is_correct)
    
    status = "CORRECT" if is_correct else "WRONG"
    print(f"{name:20} | Model: {answer:10} | Expected: {correct:5} | {status}")

few_shot_accuracy = sum(few_shot_results) / len(few_shot_results) * 100
print(f"\nFew-Shot Accuracy: {sum(few_shot_results)}/{len(few_shot_results)} = {few_shot_accuracy:.0f}%")

FEW-SHOT PROMPTING RESULTS
Barack Obama         | Model: ac         | Expected: ka    | WRONG
Taylor Swift         | Model: ft         | Expected: rt    | WRONG
Leonardo DiCaprio    | Model: di         | Expected: oo    | WRONG
Marie Curie          | Model: ie         | Expected: ee    | WRONG
Albert Einstein      | Model: n          | Expected: tn    | WRONG
Steve Jobs           | Model: bs         | Expected: es    | WRONG
Mark Zuckerberg      | Model: rk         | Expected: kg    | WRONG
Oprah Winfrey        | Model: ra         | Expected: hy    | WRONG
Warren Buffett       | Model: ff         | Expected: nt    | WRONG
Serena Williams      | Model: na         | Expected: as    | WRONG

Few-Shot Accuracy: 0/10 = 0%


### Why Few-Shot Fails

1. **Token-based processing**: LLMs process text as tokens, not individual characters
2. **Pattern ambiguity**: Without explicit reasoning, the model infers incorrect patterns
3. **No verification mechanism**: The model can't "check its work" without explicit steps

---
## Part 3: Chain-of-Thought (CoT) Reasoning

CoT prompting includes **intermediate reasoning steps** in the examples.

The model learns the algorithm, not just the pattern.

In [4]:
def cot_prompt(query: str) -> str:
    """Create a Chain-of-Thought prompt with reasoning steps."""
    return '''Q: "Elon Musk"
A: The last letter of "Elon" is "n". The last letter of "Musk" is "k". Concatenating: "nk".

Q: "Bill Gates"  
A: The last letter of "Bill" is "l". The last letter of "Gates" is "s". Concatenating: "ls".

Q: "''' + query + '''"
A: The last letter of "'''

print("Chain-of-Thought Prompt Template:")
print("-" * 40)
print(cot_prompt("[NAME]"))

Chain-of-Thought Prompt Template:
----------------------------------------
Q: "Elon Musk"
A: The last letter of "Elon" is "n". The last letter of "Musk" is "k". Concatenating: "nk".

Q: "Bill Gates"  
A: The last letter of "Bill" is "l". The last letter of "Gates" is "s". Concatenating: "ls".

Q: "[NAME]"
A: The last letter of "


In [5]:
print("CHAIN-OF-THOUGHT PROMPTING RESULTS")
print("=" * 60)

cot_results = []
for name in test_names:
    prompt = cot_prompt(name)
    response = llm.invoke(prompt).strip()
    correct = get_correct_answer(name)
    
    # Check if correct answer appears in the reasoning
    response_clean = response.lower().replace('"', '').replace(' ', '')
    is_correct = correct in response_clean
    cot_results.append(is_correct)
    
    status = "CORRECT" if is_correct else "WRONG"
    print(f"{name:20} | Expected: {correct:5} | {status}")
    # Show truncated response
    resp_short = response[:70] + "..." if len(response) > 70 else response
    print(f"  Response: {resp_short}")
    print()

cot_accuracy = sum(cot_results) / len(cot_results) * 100
print(f"CoT Accuracy: {sum(cot_results)}/{len(cot_results)} = {cot_accuracy:.0f}%")

CHAIN-OF-THOUGHT PROMPTING RESULTS
Barack Obama         | Expected: ka    | CORRECT
  Response: The last letter of "Barack" is "K". The last letter of "Obama" is "A"....

Taylor Swift         | Expected: rt    | CORRECT
  Response: The last letter of "Taylor" is "r". The last letter of "Swift" is "t"....

Leonardo DiCaprio    | Expected: oo    | CORRECT
  Response: The last letter of "Leonardo" is "o". The last letter of "DiCaprio" is...

Marie Curie          | Expected: ee    | CORRECT
  Response: The last letter of "Marie" is "e". The last letter of "Curie" is "e". ...

Albert Einstein      | Expected: tn    | CORRECT
  Response: The last letter of "Albert" is "t".
The last letter of "Einstein" is "...

Steve Jobs           | Expected: es    | CORRECT
  Response: The last letter of "Steve" is "e". The last letter of "Jobs" is "s". C...

Mark Zuckerberg      | Expected: kg    | CORRECT
  Response: The last letter of "Mark" is "k".
The last letter of "Zuckerberg" is "...

Oprah Winfrey

### Why CoT Works

1. **Explicit decomposition**: Task is broken into clear sub-steps
2. **Step-by-step execution**: Model follows a clear algorithm
3. **Verification built-in**: Each step can be verified independently
4. **Generalizable pattern**: The reasoning process transfers to new examples

---
## Key Takeaway

| Aspect | Few-Shot | CoT Reasoning |
|--------|----------|---------------|
| What's shown | Input → Output only | Input → Reasoning → Output |
| Learning | Pattern matching | Algorithmic understanding |
| Reliability | Inconsistent | More reliable |
| Generalization | Poor for algorithmic tasks | Better |

**Same task, same model, different prompting strategy = dramatically different accuracy.**

---
## Part 4: Abbreviation Task (Zero-Shot vs Few-Shot)

A simpler task: Take the **first letter** of each word to create an abbreviation.

Examples:
- "Barack Obama" → "BO"
- "Taylor Swift" → "TS"

This task is easier because it aligns better with how LLMs process tokens.

In [6]:
def get_abbreviation(name: str) -> str:
    """Ground truth: first letter of each word, uppercase."""
    return ''.join(word[0].upper() for word in name.split())

# Zero-Shot Prompt
def zero_shot_abbrev(query: str) -> str:
    return f'''Create an abbreviation by taking the first letter of each word.

Q: "{query}"
A: "'''

print("ZERO-SHOT PROMPTING - Abbreviation Task")
print("=" * 60)

zero_shot_results = []
for name in test_names:
    prompt = zero_shot_abbrev(name)
    response = llm.invoke(prompt).strip()[:20]
    correct = get_abbreviation(name)
    
    letters = re.findall(r'[a-zA-Z]+', response)
    answer = letters[0].upper() if letters else ""
    
    is_correct = answer == correct
    zero_shot_results.append(is_correct)
    
    status = "OK" if is_correct else "FAIL"
    print(f"{name:20} | Got: {answer:5} | Want: {correct:5} | {status}")

print(f"\nZero-Shot: {sum(zero_shot_results)}/{len(zero_shot_results)} = {sum(zero_shot_results)/len(zero_shot_results)*100:.0f}%")

ZERO-SHOT PROMPTING - Abbreviation Task
Barack Obama         | Got: THE   | Want: BO    | FAIL
Taylor Swift         | Got: T     | Want: TS    | FAIL
Leonardo DiCaprio    | Got: THE   | Want: LD    | FAIL
Marie Curie          | Got: THE   | Want: MC    | FAIL
Albert Einstein      | Got: AE    | Want: AE    | OK
Steve Jobs           | Got: S     | Want: SJ    | FAIL
Mark Zuckerberg      | Got: MQZ   | Want: MZ    | FAIL
Oprah Winfrey        | Got: THE   | Want: OW    | FAIL
Warren Buffett       | Got: W     | Want: WB    | FAIL
Serena Williams      | Got: THE   | Want: SW    | FAIL

Zero-Shot: 1/10 = 10%


In [7]:
# Few-Shot Prompt for Abbreviation
def few_shot_abbrev(query: str) -> str:
    return '''Q: "Elon Musk"
A: "EM"

Q: "Bill Gates"
A: "BG"

Q: "''' + query + '''"
A: "'''

print("FEW-SHOT PROMPTING - Abbreviation Task")
print("=" * 60)

few_shot_abbrev_results = []
for name in test_names:
    prompt = few_shot_abbrev(name)
    response = llm.invoke(prompt).strip()[:20]
    correct = get_abbreviation(name)
    
    letters = re.findall(r'[a-zA-Z]+', response)
    answer = letters[0].upper() if letters else ""
    
    is_correct = answer == correct
    few_shot_abbrev_results.append(is_correct)
    
    status = "OK" if is_correct else "FAIL"
    print(f"{name:20} | Got: {answer:5} | Want: {correct:5} | {status}")

print(f"\nFew-Shot: {sum(few_shot_abbrev_results)}/{len(few_shot_abbrev_results)} = {sum(few_shot_abbrev_results)/len(few_shot_abbrev_results)*100:.0f}%")

FEW-SHOT PROMPTING - Abbreviation Task
Barack Obama         | Got: BO    | Want: BO    | OK
Taylor Swift         | Got: TS    | Want: TS    | OK
Leonardo DiCaprio    | Got: LD    | Want: LD    | OK
Marie Curie          | Got: MC    | Want: MC    | OK
Albert Einstein      | Got: AE    | Want: AE    | OK
Steve Jobs           | Got: SJ    | Want: SJ    | OK
Mark Zuckerberg      | Got: MZ    | Want: MZ    | OK
Oprah Winfrey        | Got: OW    | Want: OW    | OK
Warren Buffett       | Got: WB    | Want: WB    | OK
Serena Williams      | Got: SW    | Want: SW    | OK

Few-Shot: 10/10 = 100%


In [8]:
print("\n" + "=" * 60)
print("ABBREVIATION TASK SUMMARY")
print("=" * 60)
print(f"Zero-Shot: {sum(zero_shot_results)}/{len(zero_shot_results)} = {sum(zero_shot_results)/len(zero_shot_results)*100:.0f}%")
print(f"Few-Shot:  {sum(few_shot_abbrev_results)}/{len(few_shot_abbrev_results)} = {sum(few_shot_abbrev_results)/len(few_shot_abbrev_results)*100:.0f}%")
print("\n" + "-" * 60)
print("INSIGHT: Abbreviation (first letter) is EASIER than last letter")
print("because tokens often start with the first character of words.")


ABBREVIATION TASK SUMMARY
Zero-Shot: 1/10 = 10%
Few-Shot:  10/10 = 100%

------------------------------------------------------------
INSIGHT: Abbreviation (first letter) is EASIER than last letter
because tokens often start with the first character of words.
